# Notebook 7 — Validação da Camada Semântica: Cobertura das Variáveis de Input

**Objetivo:** Verificar a cobertura (% de empresas com dado disponível) de cada variável contábil que será usada no cálculo de indicadores financeiros, identificando quais precisam de COALESCE.  
**Input:** `layer_02_silver.n1_dfp_cia_aberta_bp` · `n1_dfp_cia_aberta_dre` · `n1_dfp_cia_aberta_dfc`  
**Output:** Relatório de cobertura por variável (V01–V26) — sem escrita no banco

## 1. Setup — Bibliotecas, Conexão e Tabelas de Referência

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

def create_db_engine():
    user     = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host     = os.getenv('DB_HOST', 'localhost')
    port     = os.getenv('DB_PORT', '5432')
    dbname   = os.getenv('DB_NAME', 'data_lake')
    return create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

engine = create_db_engine()

TABELA_BP  = 'layer_02_silver.n1_dfp_cia_aberta_bp'
TABELA_DRE = 'layer_02_silver.n1_dfp_cia_aberta_dre'
TABELA_DFC = 'layer_02_silver.n1_dfp_cia_aberta_dfc'

print(f'BP  -> {TABELA_BP}')
print(f'DRE -> {TABELA_DRE}')
print(f'DFC -> {TABELA_DFC}')


BP  -> layer_02_silver.n1_dfp_cia_aberta_bp
DRE -> layer_02_silver.n1_dfp_cia_aberta_dre
DFC -> layer_02_silver.n1_dfp_cia_aberta_dfc


## 2. Função Helper de Validação — `validar_variavel()`

> Recebe um `CD_CONTA`, consulta a tabela Silver e retorna o percentual de empresas  
> com dado disponível no período mais recente. Armazena o resultado em `resultados_cobertura`.

In [2]:
def validar_variavel(
    engine,
    var_id: str,
    conceito: str,
    tabela: str,
    contas: list,
    coalesce_zero: bool = False,
    cobertura_esperada_pct: float = 95.0
):
    """
    Para uma variável, exibe:
      1. Catálogo Semântico: CD_CONTA + DS_CONTA + ST_CONTA_FIXA + IS_LEAF
         => as linhas exatas que constituem este conceito de negócio
      2. Cobertura: qtd empresas e períodos com esta conta
      3. Faixa de valores (excluindo zeros)
    """
    contas_sql = ', '.join(f"'{c}'" for c in contas)
    print(f'\n{"="*70}')
    print(f'  {var_id} — {conceito}')
    print(f'  Conta(s): {contas}   |   Tabela: {tabela.split(".")[-1]}')
    print(f'{"="*70}')

    # ── 1. CATÁLOGO SEMÂNTICO ─────────────────────────────────────────────
    q_catalogo = f"""
    SELECT
        "CD_CONTA",
        "DS_CONTA"                          AS nome_padronizado,
        "ST_CONTA_FIXA",
        "IS_LEAF",
        COUNT(*)                            AS n_linhas,
        COUNT(DISTINCT "CNPJ_CIA")          AS n_empresas,
        COUNT(DISTINCT "DT_REFER")          AS n_periodos
    FROM {tabela}
    WHERE "CD_CONTA" IN ({contas_sql})
    GROUP BY "CD_CONTA", "DS_CONTA", "ST_CONTA_FIXA", "IS_LEAF"
    ORDER BY "CD_CONTA", n_linhas DESC;
    """

    # ── 2. COBERTURA ──────────────────────────────────────────────────────
    q_cobertura = f"""
    WITH universo AS (
        SELECT COUNT(DISTINCT "CNPJ_CIA") AS total_empresas,
               COUNT(DISTINCT ("CNPJ_CIA" || "DT_REFER"::TEXT)) AS total_empresa_periodo
        FROM {tabela}
    ),
    com_conta AS (
        SELECT COUNT(DISTINCT "CNPJ_CIA") AS empresas_com_conta,
               COUNT(DISTINCT ("CNPJ_CIA" || "DT_REFER"::TEXT)) AS ep_com_conta
        FROM {tabela}
        WHERE "CD_CONTA" IN ({contas_sql})
    )
    SELECT
        u.total_empresas,
        c.empresas_com_conta,
        ROUND((c.empresas_com_conta::NUMERIC / NULLIF(u.total_empresas,0) * 100), 1) AS pct_empresas,
        u.total_empresa_periodo,
        c.ep_com_conta,
        ROUND((c.ep_com_conta::NUMERIC / NULLIF(u.total_empresa_periodo,0) * 100), 1) AS pct_ep
    FROM universo u, com_conta c;
    """

    # ── 3. FAIXA DE VALORES ───────────────────────────────────────────────
    q_valores = f"""
    SELECT
        COUNT(*)                                         AS n_total,
        SUM(CASE WHEN "VL_CONTA_TRATADO" = 0 THEN 1 ELSE 0 END) AS n_zeros,
        MIN("VL_CONTA_TRATADO")                          AS min_valor,
        PERCENTILE_CONT(0.5) WITHIN GROUP
            (ORDER BY "VL_CONTA_TRATADO")                AS mediana,
        MAX("VL_CONTA_TRATADO")                          AS max_valor
    FROM {tabela}
    WHERE "CD_CONTA" IN ({contas_sql})
      AND "IS_LEAF" = TRUE;
    """

    with engine.connect() as conn:
        df_cat = pd.read_sql(text(q_catalogo),  conn)
        df_cob = pd.read_sql(text(q_cobertura), conn)
        df_val = pd.read_sql(text(q_valores),   conn)

    print('\n📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:')
    display(df_cat)

    pct = float(df_cob['pct_empresas'].iloc[0]) if not df_cob.empty else 0
    status_cob = '✅' if pct >= cobertura_esperada_pct else ('⚠️' if pct >= 70 else '❌')
    print(f'\n📊 COBERTURA: {status_cob}')
    display(df_cob)

    print(f'\n💰 FAIXA DE VALORES (IS_LEAF = True):')
    display(df_val)

    coalesce_msg = '  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)' if coalesce_zero else ''
    if coalesce_msg:
        print(coalesce_msg)

    return pct


# Dicionário de resultados para o resumo final
resultados_cobertura = {}
print('✅ Helper definido.')


✅ Helper definido.


## 3. Variáveis do Balanço Patrimonial (BP)

> V01 a V16: contas do Ativo, Passivo e Patrimônio Líquido usadas nos indicadores de liquidez, estrutura e Fleuriet.

### V01 — Ativo Total (AT)
> `CD_CONTA = '1'` | `sem COALESCE`

In [3]:
pct = validar_variavel(engine, 'V01', 'Ativo Total (AT)', TABELA_BP, ['1'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V01'] = pct



  V01 — Ativo Total (AT)
  Conta(s): ['1']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1,ATIVO TOTAL,S,False,2220,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V02 — Ativo Circulante (AC)
> `CD_CONTA = '1.01'` | `sem COALESCE`

In [4]:
pct = validar_variavel(engine, 'V02', 'Ativo Circulante (AC)', TABELA_BP, ['1.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V02'] = pct



  V02 — Ativo Circulante (AC)
  Conta(s): ['1.01']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.01,ATIVO CIRCULANTE,S,False,2220,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V03 — Ativo Não Circulante (ANC)
> `CD_CONTA = '1.02'` | `sem COALESCE`

In [5]:
pct = validar_variavel(engine, 'V03', 'Ativo Não Circulante (ANC)', TABELA_BP, ['1.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V03'] = pct



  V03 — Ativo Não Circulante (ANC)
  Conta(s): ['1.02']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.02,ATIVO NÃO CIRCULANTE,S,False,2220,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V04 — Realizável a Longo Prazo (RLP / ELP)
> `CD_CONTA = '1.02.01'` | `sem COALESCE`

In [6]:
pct = validar_variavel(engine, 'V04', 'Realizável a Longo Prazo (RLP / ELP)', TABELA_BP, ['1.02.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V04'] = pct



  V04 — Realizável a Longo Prazo (RLP / ELP)
  Conta(s): ['1.02.01']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.02.01,ATIVO REALIZÁVEL A LONGO PRAZO,S,False,2216,185,54
1,1.02.01,ATIVO REALIZÁVEL A LONGO PRAZO,S,True,1,1,1



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2217,99.9



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1,0,2991000.0,2991000.0,2991000.0


### V05 — Ativo Imobilizado
> `CD_CONTA = '1.02.03'` | `sem COALESCE`

In [7]:
pct = validar_variavel(engine, 'V05', 'Ativo Imobilizado', TABELA_BP, ['1.02.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V05'] = pct



  V05 — Ativo Imobilizado
  Conta(s): ['1.02.03']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.02.03,IMOBILIZADO,S,False,1769,175,44
1,1.02.03,IMOBILIZADO,S,True,446,70,36



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2215,99.8



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,446,0,650975.0,481896000.0,8.439170e+11


### V06 — Estoques
> `CD_CONTA = '1.01.04'` | `COALESCE(valor, 0)`

In [8]:
pct = validar_variavel(engine, 'V06', 'Estoques', TABELA_BP, ['1.01.04'], coalesce_zero=True, cobertura_esperada_pct=85.0)
resultados_cobertura['V06'] = pct



  V06 — Estoques
  Conta(s): ['1.01.04']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.01.04,ESTOQUES,S,True,1601,144,54
1,1.01.04,ESTOQUES,S,False,355,37,16



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,165,89.2,2220,1956,88.1



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1601,0,3000.0,209591000.0,4.580400e+10


  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)


### V07 — Contas a Receber CP
> `CD_CONTA = '1.01.03'` | `sem COALESCE`

In [9]:
pct = validar_variavel(engine, 'V07', 'Contas a Receber CP', TABELA_BP, ['1.01.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V07'] = pct



  V07 — Contas a Receber CP
  Conta(s): ['1.01.03']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.01.03,CONTAS A RECEBER,S,False,2027,181,48
1,1.01.03,CONTAS A RECEBER,S,True,179,32,22



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2206,99.4



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,179,0,33658.0,909013000.0,3.553800e+10


### V09 — Fornecedores
> `CD_CONTA = '2.01.02'` | `sem COALESCE`

In [10]:
pct = validar_variavel(engine, 'V09', 'Fornecedores', TABELA_BP, ['2.01.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V09'] = pct



  V09 — Fornecedores
  Conta(s): ['2.01.02']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.01.02,FORNECEDORES,S,False,1754,173,43
1,2.01.02,FORNECEDORES,S,True,466,74,29



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,466,0,80406.0,467120000.0,3.765900e+10


### V10 — Passivo Circulante (PC)
> `CD_CONTA = '2.01'` | `sem COALESCE`

In [11]:
pct = validar_variavel(engine, 'V10', 'Passivo Circulante (PC)', TABELA_BP, ['2.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V10'] = pct



  V10 — Passivo Circulante (PC)
  Conta(s): ['2.01']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.01,PASSIVO CIRCULANTE,S,False,2220,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V11 — Passivo Não Circulante / ELP
> `CD_CONTA = '2.02'` | `sem COALESCE`

In [12]:
pct = validar_variavel(engine, 'V11', 'Passivo Não Circulante / ELP', TABELA_BP, ['2.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V11'] = pct



  V11 — Passivo Não Circulante / ELP
  Conta(s): ['2.02']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.02,PASSIVO NÃO CIRCULANTE,S,False,2218,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2218,99.9



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V12 — Patrimônio Líquido (PL)
> `CD_CONTA = '2.03'` | `sem COALESCE`

In [13]:
pct = validar_variavel(engine, 'V12', 'Patrimônio Líquido (PL)', TABELA_BP, ['2.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V12'] = pct



  V12 — Patrimônio Líquido (PL)
  Conta(s): ['2.03']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.03,PATRIMÔNIO LÍQUIDO CONSOLIDADO,S,False,2220,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2220,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,0,None,None,None,None


### V13 — Empréstimos e Financ. CP
> `CD_CONTA = '2.01.04'` | `COALESCE(valor, 0)`

In [14]:
pct = validar_variavel(engine, 'V13', 'Empréstimos e Financ. CP', TABELA_BP, ['2.01.04'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V13'] = pct



  V13 — Empréstimos e Financ. CP
  Conta(s): ['2.01.04']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.01.04,EMPRÉSTIMOS E FINANCIAMENTOS,S,False,1898,173,47
1,2.01.04,EMPRÉSTIMOS E FINANCIAMENTOS,S,True,204,32,31



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,180,97.3,2220,2102,94.7



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,204,0,607973.0,461779500.0,9.788449e+09


  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)


### V14 — Empréstimos e Financ. LP
> `CD_CONTA = '2.02.01'` | `COALESCE(valor, 0)`

In [15]:
pct = validar_variavel(engine, 'V14', 'Empréstimos e Financ. LP', TABELA_BP, ['2.02.01'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V14'] = pct



  V14 — Empréstimos e Financ. LP
  Conta(s): ['2.02.01']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.02.01,EMPRÉSTIMOS E FINANCIAMENTOS,S,False,1837,170,47
1,2.02.01,EMPRÉSTIMOS E FINANCIAMENTOS,S,True,218,33,32



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,178,96.2,2220,2055,92.6



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,218,0,465000.0,733328000.0,1.028783e+11


  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)


### V22 — Disponibilidades / Caixa BP
> `CD_CONTA = '1.01.01'` | `sem COALESCE`

In [16]:
pct = validar_variavel(engine, 'V22', 'Disponibilidades / Caixa BP', TABELA_BP, ['1.01.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V22'] = pct



  V22 — Disponibilidades / Caixa BP
  Conta(s): ['1.01.01']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,S,True,1871,171,54
1,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,S,False,346,41,15



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2217,99.9



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1871,0,-234000.0,224361000.0,9.784500e+10


### V15 — Ativo Circulante Financeiro (ACF) — Fleuriet
> `CD_CONTA IN ('1.01.01', '1.01.02')` | COALESCE para `1.01.02`

In [17]:
pct = validar_variavel(engine, 'V15', 'ACF Fleuriet (Caixa + Aplic.Fin.)', TABELA_BP,
                       ['1.01.01', '1.01.02'], coalesce_zero=True, cobertura_esperada_pct=85.0)
resultados_cobertura['V15'] = pct



  V15 — ACF Fleuriet (Caixa + Aplic.Fin.)
  Conta(s): ['1.01.01', '1.01.02']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,S,True,1871,171,54
1,1.01.01,CAIXA E EQUIVALENTES DE CAIXA,S,False,346,41,15
2,1.01.02,APLICAÇÕES FINANCEIRAS,S,False,1185,144,41
3,1.01.02,APLICAÇÕES FINANCEIRAS,S,True,221,55,18



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2220,2219,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,2092,0,-234000.0,229123000.0,9.784500e+10


  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)


### V16 — Passivo Circulante Financeiro (PCF) — Fleuriet
> Idêntico a V13: `CD_CONTA = '2.01.04'`

In [18]:
# V16 é o mesmo CD_CONTA de V13 — apenas documenta o papel semântico diferente (PCF no Fleuriet)
pct = validar_variavel(engine, 'V16', 'PCF Fleuriet (Empréstimos CP)', TABELA_BP,
                       ['2.01.04'], coalesce_zero=True, cobertura_esperada_pct=95.0)
resultados_cobertura['V16'] = pct



  V16 — PCF Fleuriet (Empréstimos CP)
  Conta(s): ['2.01.04']   |   Tabela: n1_dfp_cia_aberta_bp

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,2.01.04,EMPRÉSTIMOS E FINANCIAMENTOS,S,False,1898,173,47
1,2.01.04,EMPRÉSTIMOS E FINANCIAMENTOS,S,True,204,32,31



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,180,97.3,2220,2102,94.7



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,204,0,607973.0,461779500.0,9.788449e+09


  ℹ️  COALESCE(valor, 0) — ausência = empresa sem esta rubrica (tratar como zero)


## 4. Variáveis da DRE

> V17 a V26: contas de resultado usadas nos indicadores de lucratividade, margens e LPA.

### V17 — Receita Líquida de Vendas
> `CD_CONTA = '3.01'`

In [19]:
pct = validar_variavel(engine, 'V17', 'Receita Líquida de Vendas', TABELA_DRE, ['3.01'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V17'] = pct



  V17 — Receita Líquida de Vendas
  Conta(s): ['3.01']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.01,RECEITA DE VENDA DE BENS E/OU SERVIÇOS,S,True,1839,164,54
1,3.01,RECEITA DE VENDA DE BENS E/OU SERVIÇOS,S,False,374,36,31



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2221,2213,99.6



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1839,0,-145261000.0,1.781887e+09,6.412560e+11


### V18 — Custo dos Bens/Serviços (CPV)
> `CD_CONTA = '3.02'`

In [20]:
pct = validar_variavel(engine, 'V18', 'Custo dos Bens/Serviços (CPV)', TABELA_DRE, ['3.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V18'] = pct



  V18 — Custo dos Bens/Serviços (CPV)
  Conta(s): ['3.02']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.02,CUSTO DOS BENS E/OU SERVIÇOS VENDIDOS,S,True,1727,163,54
1,3.02,CUSTO DOS BENS E/OU SERVIÇOS VENDIDOS,S,False,465,46,21



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2221,2192,98.7



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1727,0,-3.541790e+11,-1.022330e+09,1.241950e+11


### V19 — Lucro Bruto (Resultado Bruto)
> `CD_CONTA = '3.03'`

In [21]:
pct = validar_variavel(engine, 'V19', 'Lucro Bruto (Resultado Bruto)', TABELA_DRE, ['3.03'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V19'] = pct



  V19 — Lucro Bruto (Resultado Bruto)
  Conta(s): ['3.03']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.03,RESULTADO BRUTO,S,True,2213,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2221,2213,99.6



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,2213,0,-279477000.0,522569000.0,3.341000e+11


### V20 — Lucro Operacional (EBIT)
> `CD_CONTA = '3.05'`

In [22]:
pct = validar_variavel(engine, 'V20', 'Lucro Operacional (EBIT)', TABELA_DRE, ['3.05'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V20'] = pct



  V20 — Lucro Operacional (EBIT)
  Conta(s): ['3.05']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.05,RESULTADO ANTES DO RESULTADO FINANCEIRO E DOS ...,S,True,2221,185,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2221,2221,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,2221,0,-2.833714e+10,191279000.0,2.942550e+11


### V21 — Lucro Líquido do Exercício
> `CD_CONTA = '3.11'`

In [23]:
pct = validar_variavel(engine, 'V21', 'Lucro Líquido do Exercício', TABELA_DRE, ['3.11'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V21'] = pct



  V21 — Lucro Líquido do Exercício
  Conta(s): ['3.11']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.11,LUCRO/PREJUÍZO CONSOLIDADO DO PERÍODO,S,False,2147,183,54
1,3.11,LUCRO/PREJUÍZO CONSOLIDADO DO PERÍODO,S,True,74,31,10



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,185,100.0,2221,2221,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,74,0,-815268000.0,53304500.0,9.579563e+09


### V25 — LPA Básico ON
> `CD_CONTA = '3.99.01.01'`

In [24]:
pct = validar_variavel(engine, 'V25', 'LPA Básico ON', TABELA_DRE, ['3.99.01.01'], coalesce_zero=False, cobertura_esperada_pct=80.0)
resultados_cobertura['V25'] = pct



  V25 — LPA Básico ON
  Conta(s): ['3.99.01.01']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.99.01.01,ON,N,True,1922,181,51



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,181,97.8,2221,1922,86.5



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1922,0,-2.744000e+16,0.72809,154529.0


### V26 — LPA Diluído ON
> `CD_CONTA = '3.99.02.01'`

In [25]:
pct = validar_variavel(engine, 'V26', 'LPA Diluído ON', TABELA_DRE, ['3.99.02.01'], coalesce_zero=False, cobertura_esperada_pct=80.0)
resultados_cobertura['V26'] = pct



  V26 — LPA Diluído ON
  Conta(s): ['3.99.02.01']   |   Tabela: n1_dfp_cia_aberta_dre

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,3.99.02.01,ON,N,True,1502,152,50



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,185,152,82.2,2221,1502,67.6



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,1502,0,-285444400.0,0.78287,1.148555e+07


### V26b — Cobertura COALESCE(V26, V25): estratégia real do indicador LPA Diluído
> V26 isolado = 67.6% por empresa-período. Com fallback V25: **86.7%**. Os 13.3% restantes não têm LPA de nenhum tipo (estrutural).

In [26]:
q_coalesce_lpa = """
WITH v25 AS (
    SELECT "CNPJ_CIA", "DT_REFER", "VL_CONTA_TRATADO" AS lpa_basico
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.99.01.01'
      AND "FLAG_RECONSTRUCAO" = False
      AND ABS("VL_CONTA_TRATADO") <= 10000
),
v26 AS (
    SELECT "CNPJ_CIA", "DT_REFER", "VL_CONTA_TRATADO" AS lpa_diluido
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.99.02.01'
      AND "FLAG_RECONSTRUCAO" = False
      AND ABS("VL_CONTA_TRATADO") <= 10000
),
universo AS (
    SELECT DISTINCT "CNPJ_CIA", "DT_REFER"
    FROM layer_02_silver.n1_dfp_cia_aberta_dre
    WHERE "CD_CONTA" = '3.01'
),
combinado AS (
    SELECT
        u."CNPJ_CIA",
        u."DT_REFER",
        CASE
            WHEN v26.lpa_diluido IS NOT NULL THEN 'diluido_direto'
            WHEN v25.lpa_basico  IS NOT NULL THEN 'fallback_basico'
            ELSE 'sem_lpa'
        END AS origem
    FROM universo u
    LEFT JOIN v25 ON v25."CNPJ_CIA" = u."CNPJ_CIA" AND v25."DT_REFER" = u."DT_REFER"
    LEFT JOIN v26 ON v26."CNPJ_CIA" = u."CNPJ_CIA" AND v26."DT_REFER" = u."DT_REFER"
)
SELECT
    origem,
    COUNT(*) AS qtd_empresa_periodo,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER())::NUMERIC, 1) AS pct
FROM combinado
GROUP BY origem
ORDER BY qtd_empresa_periodo DESC;
"""

with engine.connect() as conn:
    df_coalesce = pd.read_sql(text(q_coalesce_lpa), conn)

print('📊 COBERTURA REAL — COALESCE(V26, V25) por empresa-período:')
display(df_coalesce)

cobertura_coalesce_ep = float(df_coalesce.loc[df_coalesce['origem'] != 'sem_lpa', 'pct'].sum())
print(f'\n✅ Cobertura efetiva COALESCE(V26, V25): {cobertura_coalesce_ep:.1f}% dos empresa-período')
print(f'   ├── diluído direto  : {df_coalesce.loc[df_coalesce["origem"]=="diluido_direto","pct"].values[0]}%')
print(f'   ├── fallback básico : {df_coalesce.loc[df_coalesce["origem"]=="fallback_basico","pct"].values[0]}%')
print(f'   └── sem LPA (estrutural): {df_coalesce.loc[df_coalesce["origem"]=="sem_lpa","pct"].values[0]}% — irreducível')

resultados_cobertura['V26_COALESCE_EP'] = cobertura_coalesce_ep


📊 COBERTURA REAL — COALESCE(V26, V25) por empresa-período:


,origem,qtd_empresa_periodo,pct
0,diluido_direto,1494,67.5
1,fallback_basico,425,19.2
2,sem_lpa,294,13.3



✅ Cobertura efetiva COALESCE(V26, V25): 86.7% dos empresa-período
   ├── diluído direto  : 67.5%
   ├── fallback básico : 19.2%
   └── sem LPA (estrutural): 13.3% — irreducível


## 5. Variáveis da DFC

> V23: saldo final de caixa (`6.05.02`) — conceito CPC 03, inclui equivalentes até 90 dias.

### V23 — Saldo Final de Caixa (para Dívida Líquida / EV)
> `CD_CONTA = '6.05.02'` | DFC (CPC 03 — inclui CDBs/LFTs ≤90d)

In [27]:
pct = validar_variavel(engine, 'V23', 'Saldo Final Caixa DFC (6.05.02)', TABELA_DFC,
                       ['6.05.02'], coalesce_zero=False, cobertura_esperada_pct=100.0)
resultados_cobertura['V23'] = pct



  V23 — Saldo Final Caixa DFC (6.05.02)
  Conta(s): ['6.05.02']   |   Tabela: n1_dfp_cia_aberta_dfc

📋 CATÁLOGO SEMÂNTICO — linhas que constituem este conceito:


,CD_CONTA,nome_padronizado,ST_CONTA_FIXA,IS_LEAF,n_linhas,n_empresas,n_periodos
0,6.05.02,SALDO FINAL DE CAIXA E EQUIVALENTES,S,True,2179,182,54



📊 COBERTURA: ✅


,total_empresas,empresas_com_conta,pct_empresas,total_empresa_periodo,ep_com_conta,pct_ep
0,182,182,100.0,2180,2179,100.0



💰 FAIXA DE VALORES (IS_LEAF = True):


,n_total,n_zeros,min_valor,mediana,max_valor
0,2179,0,-2007000.0,239060000.0,9.784500e+10


## 6. Resumo Executivo — Cobertura por Variável

> Consolida todos os resultados de `resultados_cobertura` em um DataFrame ordenado.  
> Variáveis com cobertura < 80% devem usar `COALESCE(valor, 0)` no cálculo dos indicadores.

In [28]:
df_resumo = pd.DataFrame(
    [(k, v) for k, v in sorted(resultados_cobertura.items())],
    columns=['Variavel', 'Cobertura_Pct']
)
df_resumo['Status'] = df_resumo['Cobertura_Pct'].apply(
    lambda x: '✅ OK' if x >= 95 else ('⚠️ Parcial' if x >= 70 else '❌ Baixa')
)
df_resumo['Observacao'] = df_resumo['Variavel'].map({
    'V06': 'Serviços puros sem estoque — COALESCE(,0)',
    'V13': 'Empresa sem dívida CP — COALESCE(,0)',
    'V14': 'Empresa sem dívida LP — COALESCE(,0)',
    'V25': 'LPA Básico ON — 86.5% por empresa-período',
    'V26': 'LPA Diluído standalone — usar COALESCE(V26,V25)',
    'V26_COALESCE_EP': '✅ Cobertura REAL usada no dash (por empresa-período)',
})
df_resumo = df_resumo.sort_values('Cobertura_Pct', ascending=False)

print('\n📋 RESUMO — Cobertura das Variáveis de Input')
display(df_resumo.style.bar(subset=['Cobertura_Pct'], color='#5fba7d'))

ok   = (df_resumo['Cobertura_Pct'] >= 95).sum()
parc = ((df_resumo['Cobertura_Pct'] >= 70) & (df_resumo['Cobertura_Pct'] < 95)).sum()
baixa= (df_resumo['Cobertura_Pct'] < 70).sum()
print(f'\n✅ OK (≥95%): {ok}  |  ⚠️ Parcial (70-95%): {parc}  |  ❌ Baixa (<70%): {baixa}')
print(f'\n⚠️  V26 standalone = 82.2% (por empresa) / 67.6% (por empresa-período)')
print(f'✅  COALESCE(V26,V25) = 86.7% por empresa-período — este é o número que vai para o dash')
print(f'📌  13.3% sem LPA são estruturais (holdings/empresas sem ação ordinária) — irreducível')



📋 RESUMO — Cobertura das Variáveis de Input


,Variavel,Cobertura_Pct,Status,Observacao
0,V01,100.000000,✅ OK,nan
1,V02,100.000000,✅ OK,nan
2,V03,100.000000,✅ OK,nan
3,V04,100.000000,✅ OK,nan
4,V05,100.000000,✅ OK,nan
6,V07,100.000000,✅ OK,nan
7,V09,100.000000,✅ OK,nan
8,V10,100.000000,✅ OK,nan
9,V11,100.000000,✅ OK,nan
21,V23,100.000000,✅ OK,nan



✅ OK (≥95%): 22  |  ⚠️ Parcial (70-95%): 3  |  ❌ Baixa (<70%): 0

⚠️  V26 standalone = 82.2% (por empresa) / 67.6% (por empresa-período)
✅  COALESCE(V26,V25) = 86.7% por empresa-período — este é o número que vai para o dash
📌  13.3% sem LPA são estruturais (holdings/empresas sem ação ordinária) — irreducível
